<a href="https://colab.research.google.com/github/ZeshanRasul/RL_Experiments/blob/main/Cart_Control_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import gymnasium as gym

In [ ]:
import random

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

In [ ]:
def basic_policy(obs):
  angle = obs[2]
  return 0 if angle < 0 else 1

In [ ]:
totals = []
for episode in range(500):
  episode_rewards = 0
  obs, info = env.reset(seed=episode)
  for step in range(200):
    action = basic_policy(obs)
    obs, reward, done, truncated, info = env.step(action)
    episode_rewards += reward
    if done or truncated:
      break

  totals.append(episode_rewards)

In [ ]:
import numpy as np

In [ ]:
np.mean(totals), np.std(totals), np.min(totals), np.max(totals)

(np.float64(41.698),
 np.float64(8.389445512070509),
 np.float64(24.0),
 np.float64(63.0))

In [ ]:
import tensorflow as tf

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(5, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

In [ ]:
def play_one_step(env, obs, model, loss_fn):
  with tf.GradientTape() as tape:
    left_proba = model(obs[np.newaxis])
    action = (tf.random.uniform([1, 1]) > left_proba)
    y_target = tf.constant([[1.]]) - tf.cast(action, tf.float32)
    loss = tf.reduce_mean(loss_fn(y_target, left_prob))

  grads = tape.gradients(loss, model.trainable_variables)
  obs, rewards, done, truncated, info = env.step(int(action))
  return obs, reward, done, truncated, grads

In [ ]:
def play_multiple_episodes(env, n_episodes, n_max_steps, model, loss_fn):
  all_rewards = []
  all_grads = []
  for episode in range(n_episodes):
    current_rewards = []
    current_grads = []
    obs, info = env.reset()
    for step in range(n_max_steps):
      obs, rewards, done, truncated, grads = play_one_step(env, obs, model, loss_fn)
      current_rewards.append(rewards)
      current_grads.append(grads)
      if done or truncated:
        break

    all_rewards.append(current_rewards)
    all_grads.append(current_grads)

  return all_rewards, all_gradds

In [ ]:
def discount_rewards(rewards, discount_factor):
  discounted = np.array(rewards)
  for step in range(len(rewards) - 2, -1, -1):
    discounted[step] += discounted[step + 1] * discount_factor
  return discounted

In [ ]:
def discount_and_normalize_rewards(all_rewards, discount_factor):
  all_discounted_rewards = [discount_rewards(rewards, discount_factor) for rewards in all_rewards]
  flat_rewards = np.concatenate(all_discounted_rewards)
  reward_mean = flat_rewards.mean()
  reward_std = flat_rewards.std()
  return [(discounted_rewards - reward_mean) / reward_std for discounted_rewards in all_discounted_rewards]

In [ ]:
discount_rewards([10, 0, -50], 0.8)

array([-22, -40, -50])

In [ ]:
discount_and_normalize_rewards([[10, 0, -50], [10, 20]], discount_factor=0.8)

[array([-0.28435071, -0.86597718, -1.18910299]),
 array([1.26665318, 1.0727777 ])]

In [ ]:
n_iterations = 150
n_episodes_per_update = 10
n_max_steps = 200
discount_factor = 0.95

In [ ]:
optimzer = tf.keras.optimizers.Nadam(learning_rate=0.01)
loss_fn = tf.keras.losses.binary_crossentropy

In [ ]:
for iteration in range(n_iterations):
  all_rewards, all_grads = play_multiple_episodes(
      env, n_episodes_per_update, n_max_steps, model, loss_fun
  )
  all_final_rewards = discount_and_normalize_rewards(all_rewards, discount_factor)
  all_mean_grads = []
  for var_index in range(len(model.trainable_variables)):
    mean_grads = tf.reduce_mean(
        [final_reward * all_grads[episode_index][step][var_index]
                                  for episode_index, final rewards in enumerate(all_final_rewards)
                                    for step, final_reward in enumerate(final_rewards)], axis=0)

    all_mean_grads.append(mean_grads)

  optimizer.apply_gradients(zip(all_mean_grads, model.training_variables))
